In [44]:
import requests, pandas as pd
from io import StringIO

r = requests.get(
    "https://data.cityofnewyork.us/resource/76xm-jjuj.csv",
    params={"$limit": 100000},
    headers={"X-App-Token": "rMrnt2613XPmKlC6g4YHBYkgc"},  # optional
    timeout=60,
)
r.raise_for_status()
df = pd.read_csv(StringIO(r.text))

In [45]:

df = df.drop(columns=["incident_id", "reopen_indicator", "special_event_indicator", "standby_indicator", "transfer_indicator", "congressionaldistrict", "communityschooldistrict", "citycouncildistrict", "policeprecinct", "incident_disposition_code", "incident_dispatch_area", "communitydistrict"])

In [46]:
dt_cols = ["incident_datetime", "first_assignment_datetime", "first_on_scene_datetime"]
for c in dt_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

num_cols = ["incident_response_seconds_qy", "incident_travel_tm_seconds_qy"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [47]:
df.head()

,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,first_assignment_datetime,valid_dispatch_rspns_time_indc,dispatch_response_seconds_qy,first_activation_datetime,first_on_scene_datetime,valid_incident_rspns_time_indc,incident_response_seconds_qy,incident_travel_tm_seconds_qy,first_to_hosp_datetime,first_hosp_arrival_datetime,incident_close_datetime,held_indicator,borough,zipcode
0,2005-01-01 00:00:24,INJURY,5,INJURY,5,2005-01-01 00:04:11,Y,227,2005-01-01T00:06:15.000,2005-01-01 00:16:06,Y,942.0,715.0,2005-01-01T00:26:53.000,2005-01-01T00:39:48.000,2005-01-01T01:04:36.000,N,BROOKLYN,11217.0
1,2005-01-01 00:00:25,OTHER,6,CVAC,2,2005-01-01 00:17:13,Y,1008,2005-01-01T00:18:06.000,2005-01-01 00:37:21,Y,2216.0,1208.0,2005-01-01T00:50:30.000,2005-01-01T01:04:58.000,2005-01-01T01:44:41.000,Y,BRONX,10459.0
2,2005-01-01 00:00:43,RESPIR,4,RESPIR,4,2005-01-01 00:01:09,Y,26,2005-01-01T00:02:44.000,2005-01-01 00:08:03,Y,440.0,414.0,2005-01-01T00:21:05.000,2005-01-01T00:26:40.000,2005-01-01T00:57:18.000,N,BRONX,10462.0
3,2005-01-01 00:00:37,OBMIS,4,OBMIS,4,2005-01-01 00:02:55,Y,138,2005-01-01T00:06:22.000,2005-01-01 00:06:50,Y,373.0,235.0,2005-01-01T00:32:30.000,2005-01-01T00:42:34.000,2005-01-01T01:07:58.000,N,MANHATTAN,10019.0
4,2005-01-01 00:00:58,INJMIN,7,INJMIN,7,2005-01-01 00:02:34,Y,96,2005-01-01T00:03:24.000,2005-01-01 00:19:48,Y,1130.0,1034.0,NaN,NaN,2005-01-01T00:22:58.000,N,BROOKLYN,11212.0


In [48]:
import pandas as pd, numpy as np

df["incident_datetime"] = pd.to_datetime(df["incident_datetime"], errors="coerce")
df = df.dropna(subset=["incident_datetime", "zipcode"])
df["bucket"] = df["incident_datetime"].dt.floor("30min")

counts = df.groupby(["zipcode", "bucket"]).size().rename("y")

# Full grid so quiet periods appear as zeros
zones = df["zipcode"].unique()
buckets = pd.date_range(df["bucket"].min(), df["bucket"].max(), freq="30min")
idx = pd.MultiIndex.from_product([zones, buckets], names=["zipcode", "bucket"])
panel = counts.reindex(idx, fill_value=0).reset_index()
panel = panel.sort_values(["zipcode", "bucket"]).reset_index(drop=True)

In [49]:
g = panel.groupby("zipcode")["y"]

for lag in [1, 2, 3, 6, 48, 336]:        # 30 min, 1h, 1.5h, 3h, 1 day, 1 week
    panel[f"lag_{lag}"] = g.shift(lag)

panel["roll_2h"]  = g.transform(lambda s: s.shift(1).rolling(4).sum())
panel["roll_6h"]  = g.transform(lambda s: s.shift(1).rolling(12).sum())
panel["roll_24h"] = g.transform(lambda s: s.shift(1).rolling(48).sum())

t = panel["bucket"]
panel["hour"] = t.dt.hour + t.dt.minute / 60
panel["dow"] = t.dt.dayofweek
panel["month"] = t.dt.month
panel["is_weekend"] = (panel["dow"] >= 5).astype(int)
panel["zipcode"] = panel["zipcode"].astype("category")

In [50]:
train_end = "2022-12-31"
train = panel[panel["bucket"] <= train_end]
test = panel[panel["bucket"] > train_end]

hist = (train.groupby(["zipcode", "dow", "hour"], observed=True)["y"]
              .mean().rename("baseline").reset_index())
test = test.merge(hist, on=["zipcode", "dow", "hour"], how="left")

In [62]:
import lightgbm as lgb

features = [c for c in panel.columns if c not in ["y", "bucket"]]
train_df = train.dropna(subset=["lag_336"])      # drop rows without full history
start, end = panel["bucket"].min(), panel["bucket"].max()
span = end - start
train_end = start + span * 0.75
val_cut   = start + span * 0.60

train = panel[panel["bucket"] <= train_end]
test  = panel[panel["bucket"] >  train_end]

train_df = train.dropna(subset=["lag_336"])
tr = train_df[train_df["bucket"] <  val_cut]
va = train_df[train_df["bucket"] >= val_cut]
print(tr.shape, va.shape, test.shape)   # all three should be non-empty

model = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=2000, learning_rate=0.03,
    num_leaves=63, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
)
model.fit(
    tr[features], tr["y"],
    eval_X=va[features], 
    eval_y=va["y"],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)
test["pred"] = model.predict(test[features])

(132000, 16) (51480, 16) (85800, 16)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010140 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 444
[LightGBM] [Info] Number of data points in the train set: 132000, number of used features: 13
[LightGBM] [Info] Start training from score -1.320138
Training until validation scores don't improve for 100 rounds
[200]	valid_0's poisson: 0.49269
[400]	valid_0's poisson: 0.490427
Early stopping, best iteration is:
[376]	valid_0's poisson: 0.490378


In [61]:
print(panel["bucket"].min(), panel["bucket"].max())
print(panel.shape)
print(panel.isna().sum().sort_values(ascending=False).head(8))
print(train.shape, train_df.shape, tr.shape, va.shape)

2005-01-01 00:00:00 2005-02-02 11:30:00
(343200, 16)
lag_336     73920
lag_48      10560
roll_24h    10560
roll_6h      2640
lag_6        1320
roll_2h       880
lag_3         660
lag_2         440
dtype: int64
(343200, 16) (269280, 16) (149160, 16) (120120, 16)


In [64]:
hist = (train.groupby(["zipcode", "dow", "hour"], observed=True)["y"]
              .mean().rename("baseline").reset_index())

test = test.merge(hist, on=["zipcode", "dow", "hour"], how="left")

In [65]:
from sklearn.metrics import mean_poisson_deviance, mean_absolute_error

t2 = test.dropna(subset=["pred", "baseline"])
print("MAE  model:", mean_absolute_error(t2.y, t2.pred),
      " baseline:", mean_absolute_error(t2.y, t2.baseline))
print("Poisson deviance model:", mean_poisson_deviance(t2.y, t2.pred.clip(1e-6)),
      " baseline:", mean_poisson_deviance(t2.y, t2.baseline.clip(1e-6)))

MAE  model: 0.34061848333392075  baseline: 0.3557886557886558
Poisson deviance model: 0.6437258992803876  baseline: 2.242235323119212
